# VideoSeal results for three MOSE samples

VideoSeal watermark embedding, object removal, and derived spatial localization are shown for three fixed MOSE samples.

```text
original -> VideoSeal embedding -> watermarked video -> ProPainter or DiffuEraser -> derived localization
```

VideoSeal can watermark an existing video. The released checkpoint provides a global watermark-presence detector rather than a native pixel-level tamper map. The evaluation therefore divides each frame into a 4 x 4 crop grid and derives an absence mask. White predicted pixels indicate crops where the detector did not find the watermark. The result must be described as a derived crop-grid proxy, not native VideoSeal localization.


## Metric guide

| Term | Meaning | Interpretation |
| --- | --- | --- |
| Global pixel IoU | Total intersection divided by total union across all frames and pixels. | 1 is perfect overlap. 0 means no overlap. This is the primary localization result. |
| Mean frame IoU | IoU calculated for each frame and then averaged. | Empty ground-truth and prediction masks score 1, so this value can look high when many frames have no target. |
| Effective GT mask | The removal mask after the remover's actual preprocessing. | This is the correct reference mask for IoU. |
| Predicted mask | The derived 4 x 4 crop-grid absence prediction. | White indicates predicted tampering. Black indicates no predicted tampering. |
| Intersection pixels | Pixels marked by both the prediction and effective ground truth. | Used in the numerator of global IoU. |
| Union pixels | Pixels marked by either mask. | Used in the denominator of global IoU. |
| Runtime | Wall-clock time recorded on the experiment hardware. | Useful for this run, not a hardware-independent benchmark. |
| SHA-256 | File fingerprint stored in the JSON record. | Confirms exact file identity. It is not a quality metric. |


## Notebook setup

After a fresh clone, run `python3 scripts/prepare_mose_notebooks.py` from the repository root. This downloads and verifies the 48 videos shared by the three method notebooks. Run the next cell once to enable the video, image, download, and metadata viewers. If Git LFS has left a pointer file in place of a video, the viewer reports the preparation command instead of displaying a broken player.


In [ ]:
import base64
from pathlib import Path
import html
import json
from IPython.display import HTML, Markdown, display

LFS_POINTER_PREFIX = b"version https://git-lfs.github.com/spec/v1"

def find_repo_root():
    for start in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (start / "evidence/mose-task1/review50/manifest.json").is_file():
            return start
    raise FileNotFoundError("Run this notebook from inside the paper-reproductions checkout.")

REPO_ROOT = find_repo_root()
REVIEW_ROOT = REPO_ROOT / "evidence/mose-task1/review50"

def artifact_data_url(source, mime_type):
    if not source.is_file():
        raise FileNotFoundError(source)
    payload = source.read_bytes()
    if payload.startswith(LFS_POINTER_PREFIX):
        raise RuntimeError(
            f"{source} is a Git LFS pointer, not the real media file. "
            "From the repository root run: python3 scripts/prepare_mose_notebooks.py"
        )
    return f"data:{mime_type};base64,{base64.b64encode(payload).decode('ascii')}"

def show_videos(items, columns=3):
    cards = []
    for title, relative, explanation in items:
        if relative is None:
            cards.append(
                f"<section class='media-card'><h4>{html.escape(title)}</h4>"
                f"<p><strong>Not applicable.</strong></p><p>{html.escape(explanation)}</p></section>"
            )
            continue
        source = REVIEW_ROOT / relative
        data_url = artifact_data_url(source, "video/mp4")
        cards.append(
            "<section class='media-card'>"
            f"<h4>{html.escape(title)}</h4>"
            f"<video controls muted loop playsinline preload='metadata' src='{data_url}'></video>"
            f"<p>{html.escape(explanation)}</p>"
            f"<p><a href='{data_url}' download='{html.escape(source.name)}'>Download this video</a></p>"
            "</section>"
        )
    display(HTML(
        f"<style>.media-grid{{display:grid;grid-template-columns:repeat({columns},minmax(220px,1fr));gap:16px}}"
        ".media-card{border:1px solid #d0d7de;border-radius:8px;padding:12px;background:#fff;color:#24292f}"
        ".media-card video{width:100%;background:#000}.media-card h4{margin:0 0 8px}.media-card p{font-size:13px}</style>"
        f"<div class='media-grid'>{''.join(cards)}</div>"
    ))

def show_images(items, columns=2):
    cards = []
    for title, relative, explanation in items:
        source = REVIEW_ROOT / relative
        data_url = artifact_data_url(source, "image/png")
        cards.append(
            "<section class='media-card'>"
            f"<h4>{html.escape(title)}</h4>"
            f"<img style='width:100%' src='{data_url}' alt='{html.escape(title)}'>"
            f"<p>{html.escape(explanation)}</p>"
            f"<p><a href='{data_url}' download='{html.escape(source.name)}'>Download this image</a></p>"
            "</section>"
        )
    display(HTML(
        f"<div class='media-grid' style='grid-template-columns:repeat({columns},minmax(260px,1fr))'>"
        f"{''.join(cards)}</div>"
    ))

def show_metadata(paths):
    for relative in paths:
        path = REVIEW_ROOT / relative
        payload = json.loads(path.read_text())
        display(Markdown(f"#### `{relative}`"))
        print(json.dumps(payload, indent=2))

print(f"Loaded evidence from {REVIEW_ROOT}")


## 1. Sample `02b72b8d`


### Source and unwatermarked removal reference

The source has 67 frames at 960 x 480, 25 FPS, and a duration of about 2.68 seconds. The raw mask marks the object selected for removal. The two unwatermarked outputs show how ProPainter and DiffuEraser behave before watermarking is introduced.

Direct links: [original](../../evidence/mose-task1/review50/media/02b72b8d/original.mp4) | [raw mask](../../evidence/mose-task1/review50/media/02b72b8d/gt_mask.mp4) | [ProPainter reference](../../evidence/mose-task1/review50/media/02b72b8d/propainter.mp4) | [DiffuEraser reference](../../evidence/mose-task1/review50/media/02b72b8d/diffueraser.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/02b72b8d/original.mp4', 'Source video before watermarking or removal.'), ('Raw selected-object mask', 'media/02b72b8d/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter without watermark', 'media/02b72b8d/propainter.mp4', 'Unwatermarked ProPainter removal reference.'), ('DiffuEraser without watermark', 'media/02b72b8d/diffueraser.mp4', 'Unwatermarked DiffuEraser removal reference.')])


### VideoSeal results

VideoSeal embedded a 256-bit message into 67 frames. Embedding took 2.284 s. The detector threshold was `0.0`. A positive detector score means watermark present. A score at or below the threshold becomes predicted tampering.

| Metric | ProPainter | DiffuEraser |
| --- | ---: | ---: |
| Removal runtime | 31.445 s | 77.269 s |
| Frames | 67 | 67 |
| Output dimensions | 960 x 480 | 960 x 480 |
| Global pixel IoU | **0.0000** | **0.0000** |
| Mean frame IoU | 0.0000 | 0.0000 |
| Minimum frame IoU | 0.0000 | 0.0000 |
| Maximum frame IoU | 0.0000 | 0.0000 |
| Intersection pixels | 0 | 0 |
| Union pixels | 774,798 | 828,241 |

Localization for both attacks took 7.587 s in total.

**Interpretation:** The derived proxy is evaluated against the effective mask used by each remover. A global IoU of 0 means that no predicted tamper pixels overlapped the actual removal region.


In [ ]:
show_videos([('VideoSeal watermarked', 'media/02b72b8d/methods/videoseal/watermarked.mp4', 'Watermarked video before object removal.'), ('ProPainter attacked video', 'media/02b72b8d/methods/videoseal/propainter.mp4', 'ProPainter output from the VideoSeal-watermarked input.'), ('ProPainter effective GT', 'media/02b72b8d/methods/videoseal/effective-gt-mask-propainter.mp4', 'Mask after ProPainter preprocessing. This is its IoU reference.'), ('ProPainter predicted mask', 'media/02b72b8d/methods/videoseal/pred-mask-propainter.mp4', 'Derived VideoSeal crop-grid prediction after ProPainter.'), ('DiffuEraser attacked video', 'media/02b72b8d/methods/videoseal/diffueraser.mp4', 'DiffuEraser output from the VideoSeal-watermarked input.'), ('DiffuEraser effective GT', 'media/02b72b8d/methods/videoseal/effective-gt-mask-diffueraser.mp4', 'Mask after DiffuEraser preprocessing. This is its IoU reference.'), ('DiffuEraser predicted mask', 'media/02b72b8d/methods/videoseal/pred-mask-diffueraser.mp4', 'Derived VideoSeal crop-grid prediction after DiffuEraser.')])


### Run records

The next cell prints the embedding, removal, and localization JSON records. The localization record includes per-frame IoU values and file fingerprints.


In [ ]:
show_metadata(['media/02b72b8d/methods/videoseal/diffueraser.json', 'media/02b72b8d/methods/videoseal/embed.json', 'media/02b72b8d/methods/videoseal/localization.json', 'media/02b72b8d/methods/videoseal/propainter.json'])


## 2. Sample `0ab75563`


### Source and unwatermarked removal reference

The source has 104 frames at 960 x 448, 25 FPS, and a duration of about 4.16 seconds. The raw mask marks the object selected for removal. The two unwatermarked outputs show how ProPainter and DiffuEraser behave before watermarking is introduced.

Direct links: [original](../../evidence/mose-task1/review50/media/0ab75563/original.mp4) | [raw mask](../../evidence/mose-task1/review50/media/0ab75563/gt_mask.mp4) | [ProPainter reference](../../evidence/mose-task1/review50/media/0ab75563/propainter.mp4) | [DiffuEraser reference](../../evidence/mose-task1/review50/media/0ab75563/diffueraser.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/0ab75563/original.mp4', 'Source video before watermarking or removal.'), ('Raw selected-object mask', 'media/0ab75563/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter without watermark', 'media/0ab75563/propainter.mp4', 'Unwatermarked ProPainter removal reference.'), ('DiffuEraser without watermark', 'media/0ab75563/diffueraser.mp4', 'Unwatermarked DiffuEraser removal reference.')])


### VideoSeal results

VideoSeal embedded a 256-bit message into 104 frames. Embedding took 3.231 s. The detector threshold was `0.0`. A positive detector score means watermark present. A score at or below the threshold becomes predicted tampering.

| Metric | ProPainter | DiffuEraser |
| --- | ---: | ---: |
| Removal runtime | 43.633 s | 98.975 s |
| Frames | 104 | 104 |
| Output dimensions | 960 x 448 | 960 x 448 |
| Global pixel IoU | **0.0000** | **0.0000** |
| Mean frame IoU | 0.8365 | 0.8365 |
| Minimum frame IoU | 0.0000 | 0.0000 |
| Maximum frame IoU | 1.0000 | 1.0000 |
| Intersection pixels | 0 | 0 |
| Union pixels | 490,646 | 499,284 |

Localization for both attacks took 10.619 s in total.

**Interpretation:** The derived proxy is evaluated against the effective mask used by each remover. A global IoU of 0 means that no predicted tamper pixels overlapped the actual removal region. The mean frame IoU is raised by 87 frames where both masks are empty. Those empty frames score 1 even though the detector did not localize the edited region.


In [ ]:
show_videos([('VideoSeal watermarked', 'media/0ab75563/methods/videoseal/watermarked.mp4', 'Watermarked video before object removal.'), ('ProPainter attacked video', 'media/0ab75563/methods/videoseal/propainter.mp4', 'ProPainter output from the VideoSeal-watermarked input.'), ('ProPainter effective GT', 'media/0ab75563/methods/videoseal/effective-gt-mask-propainter.mp4', 'Mask after ProPainter preprocessing. This is its IoU reference.'), ('ProPainter predicted mask', 'media/0ab75563/methods/videoseal/pred-mask-propainter.mp4', 'Derived VideoSeal crop-grid prediction after ProPainter.'), ('DiffuEraser attacked video', 'media/0ab75563/methods/videoseal/diffueraser.mp4', 'DiffuEraser output from the VideoSeal-watermarked input.'), ('DiffuEraser effective GT', 'media/0ab75563/methods/videoseal/effective-gt-mask-diffueraser.mp4', 'Mask after DiffuEraser preprocessing. This is its IoU reference.'), ('DiffuEraser predicted mask', 'media/0ab75563/methods/videoseal/pred-mask-diffueraser.mp4', 'Derived VideoSeal crop-grid prediction after DiffuEraser.')])


### Run records

The next cell prints the embedding, removal, and localization JSON records. The localization record includes per-frame IoU values and file fingerprints.


In [ ]:
show_metadata(['media/0ab75563/methods/videoseal/diffueraser.json', 'media/0ab75563/methods/videoseal/embed.json', 'media/0ab75563/methods/videoseal/localization.json', 'media/0ab75563/methods/videoseal/propainter.json'])


## 3. Sample `0b02886a`


### Source and unwatermarked removal reference

The source has 30 frames at 960 x 640, 25 FPS, and a duration of about 1.20 seconds. The raw mask marks the object selected for removal. The two unwatermarked outputs show how ProPainter and DiffuEraser behave before watermarking is introduced.

Direct links: [original](../../evidence/mose-task1/review50/media/0b02886a/original.mp4) | [raw mask](../../evidence/mose-task1/review50/media/0b02886a/gt_mask.mp4) | [ProPainter reference](../../evidence/mose-task1/review50/media/0b02886a/propainter.mp4) | [DiffuEraser reference](../../evidence/mose-task1/review50/media/0b02886a/diffueraser.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/0b02886a/original.mp4', 'Source video before watermarking or removal.'), ('Raw selected-object mask', 'media/0b02886a/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter without watermark', 'media/0b02886a/propainter.mp4', 'Unwatermarked ProPainter removal reference.'), ('DiffuEraser without watermark', 'media/0b02886a/diffueraser.mp4', 'Unwatermarked DiffuEraser removal reference.')])


### VideoSeal results

VideoSeal embedded a 256-bit message into 30 frames. Embedding took 0.719 s. The detector threshold was `0.0`. A positive detector score means watermark present. A score at or below the threshold becomes predicted tampering.

| Metric | ProPainter | DiffuEraser |
| --- | ---: | ---: |
| Removal runtime | 20.083 s | 47.145 s |
| Frames | 30 | 30 |
| Output dimensions | 960 x 640 | 960 x 640 |
| Global pixel IoU | **0.0000** | **0.0000** |
| Mean frame IoU | 0.0000 | 0.0000 |
| Minimum frame IoU | 0.0000 | 0.0000 |
| Maximum frame IoU | 0.0000 | 0.0000 |
| Intersection pixels | 0 | 0 |
| Union pixels | 1,721,722 | 1,767,014 |

Localization for both attacks took 3.279 s in total.

**Interpretation:** The derived proxy is evaluated against the effective mask used by each remover. A global IoU of 0 means that no predicted tamper pixels overlapped the actual removal region.


In [ ]:
show_videos([('VideoSeal watermarked', 'media/0b02886a/methods/videoseal/watermarked.mp4', 'Watermarked video before object removal.'), ('ProPainter attacked video', 'media/0b02886a/methods/videoseal/propainter.mp4', 'ProPainter output from the VideoSeal-watermarked input.'), ('ProPainter effective GT', 'media/0b02886a/methods/videoseal/effective-gt-mask-propainter.mp4', 'Mask after ProPainter preprocessing. This is its IoU reference.'), ('ProPainter predicted mask', 'media/0b02886a/methods/videoseal/pred-mask-propainter.mp4', 'Derived VideoSeal crop-grid prediction after ProPainter.'), ('DiffuEraser attacked video', 'media/0b02886a/methods/videoseal/diffueraser.mp4', 'DiffuEraser output from the VideoSeal-watermarked input.'), ('DiffuEraser effective GT', 'media/0b02886a/methods/videoseal/effective-gt-mask-diffueraser.mp4', 'Mask after DiffuEraser preprocessing. This is its IoU reference.'), ('DiffuEraser predicted mask', 'media/0b02886a/methods/videoseal/pred-mask-diffueraser.mp4', 'Derived VideoSeal crop-grid prediction after DiffuEraser.')])


### Run records

The next cell prints the embedding, removal, and localization JSON records. The localization record includes per-frame IoU values and file fingerprints.


In [ ]:
show_metadata(['media/0b02886a/methods/videoseal/diffueraser.json', 'media/0b02886a/methods/videoseal/embed.json', 'media/0b02886a/methods/videoseal/localization.json', 'media/0b02886a/methods/videoseal/propainter.json'])


## Validation and provenance

- The committed validator decoded the videos, checked frame counts, dimensions, and FPS, and verified the required method fields.
- Aggregate and per-file validation: [validation.json](../../evidence/mose-task1/review50/validation.json)
- Aggregate results: [watermark-summary.json](../../evidence/mose-task1/review50/watermark-summary.json)
- Experimental contract: [PROTOCOL.md](../../evidence/mose-task1/PROTOCOL.md)
- Batch summary: [README.md](../../evidence/mose-task1/README.md)

The displayed values come from the committed VideoSeal evidence records.
